In [ ]:
import runpy
import shutil
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

import fluidbench
from fluidbench.cfdataset import CFDataset

In [ ]:
ID = "cylinder_interp"
dataset_path = fluidbench.download(ID)

In [ ]:
def load_split(split):
    folder = dataset_path / split
    cases = sorted((path for path in folder.glob("*.h5") if path.stem != "constants"), key=lambda path: int(path.stem))
    data = []
    for case in cases:
        with h5py.File(case, "r") as file:
            data.append((file["scalars"][:], file["fields"][:]))
    return data

def to_dataset(cases):
    # Each snapshot is paired with the next one within its case: inputs are snapshot i, targets snapshot i + 1.
    scalars = np.concatenate([scalars[:-1] for scalars, fields in cases])
    inputs = np.concatenate([fields[:-1] for scalars, fields in cases])
    targets = np.concatenate([fields[1:] for scalars, fields in cases])
    return CFDataset(scalars, inputs, targets, np.arange(len(targets)))

In [ ]:
# The benchmark ships train and test only. With many cases, hold out whole random cases (Reynolds numbers)
# for validation: it measures generalisation to unseen Re and avoids near-duplicate snapshots leaking across.
VAL_FRACTION = 0.1
cases = load_split("train")
order = np.random.default_rng(0).permutation(len(cases))
n_validation = max(1, round(len(cases) * VAL_FRACTION))
validation_cases = [cases[i] for i in order[:n_validation]]
train_cases = [cases[i] for i in order[n_validation:]]

train_dataset = to_dataset(train_cases)
validation_dataset = to_dataset(validation_cases)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
print(f"{len(train_cases)} train cases ({len(train_dataset)} pairs), {len(validation_cases)} validation cases ({len(validation_dataset)} pairs)")
print("Validation Re:", sorted(float(scalars[0, 0]) for scalars, _ in validation_cases))

In [ ]:
def train_and_bench(architecture, submission_id, name):
    model = architecture()
    model.to(model.device)
    history = model.fit(train_loader, validation_loader)

    weights_path = Path("weights") / ID / f"{architecture.__name__}.pt"
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(weights_path)

    submission_config = {
        "datasetId": ID,
        "id": submission_id,
        "model": name,
        "authors": "Pablo A.C.",
        "year": 2026,
        "representation": "Grid",
    }
    training_history = {
        "train_error_history": history["train_losses"],
        "val_error_history": history["val_losses"],
        "train_error_name": history["loss_name"],
        "val_split": VAL_FRACTION,
    }
    return fluidbench.run(model, submission_config, training_history)

In [ ]:
from models.minimal_models import BaselineSame
submission = train_and_bench(BaselineSame, "baseline-same", "Baseline Same")
submission["metrics"]["rolloutRmse"]

In [ ]:
from models.minimal_models import BaselineZero
submission = train_and_bench(BaselineZero, "baseline-zero", "Baseline Zero")
submission["metrics"]["rolloutRmse"]

In [ ]:
from models.unet import UNetClassic
submission = train_and_bench(UNetClassic, "unet-classic", "UNet Classic")
submission["metrics"]["rolloutRmse"]

In [ ]:
# Publish: copy this benchmark's submissions into the site's submissions folder and refresh its manifest.
published = Path("..") / "submissions" / ID
shutil.copytree(Path("submission") / ID, published, dirs_exist_ok=True)
runpy.run_path(str(Path("..") / "submissions" / "update.py"), run_name="__main__")
sorted(path.name for path in published.glob("*.json"))